# PDE

---

Suppose S1(t) and S2(t) are independent stock processes with equal volatility, S1(0) = S2(0) = 100.

The contract at maturity pays the value of the most expensive stock: V (T) = max{S1(T), S2(T)}.

a.) What is the formula for the fair value of the contract?

b.) Derive the partial differential equation for this contract

c.) Solve the resultant PDE using Finite Difference Method

d.) Compare the results obtained through solving the PDE with Monte Carlo Method

## Imports and constants

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Model parameters
S0 = 100.0
sigma = 0.20
r = 0.05 # risk-free rate
T = 1.0 # maturity in years

# Monte Carlo parameters
np.random.seed(42)
N_sims = 500000

## Part a.) Deriving the formula for the fair value of the contract

https://en.wikipedia.org/wiki/Margrabe%27s_formula

Rewrite the maximum function:
$$\max(S_1, S_2) = S_1 + \max(S_2 - S_1, 0)$$
This exchange option can be priced using Margrabe's Formula. The current value of the contract is the discounted expected value:
$$V(0) = S_1(0) + \text{Margrabe}(S_1, S_2, T)$$

Since $S_1$ and $S_2$ are independent, the relative volatility between the two stocks is $\sigma_{rel} = \sqrt{\sigma^2 + \sigma^2} = \sigma\sqrt{2}$.

Based on Margrabe's formula:
$$\text{Margrabe} = S_0 N(d_1) - S_0 N(d_2)$$
where:$$d_1 = \frac{\ln(S_0/S_0) + \frac{1}{2}\sigma_{rel}^2 T}{\sigma_{rel}\sqrt{T}} = \frac{\sigma^2 T}{\sigma\sqrt{2T}} = \sigma\sqrt{\frac{T}{2}}$$
$$d_2 = d_1 - \sigma \sqrt{T}$$

Adding back the $S_1(0)$ term (which = $S_0$), the analytical formula becomes:
$$V(0) = S_0 + S_0 [N(d_1) - N(-d_1)]$$

Since $1 - N(-d_1) = N(d_1)$, this simplifies to:
$$V(0) = 2 S_0 N\left(\sigma\sqrt{\frac{T}{2}}\right)$$
where $N$ is the standard-normal CDF.


## Part b.) Deriving the PDE

The standard 2D Black-Scholes PDE for two assets is:
$$\frac{\partial V}{\partial t} + rS_1\frac{\partial V}{\partial S_1} + rS_2\frac{\partial V}{\partial S_2} + \frac{1}{2}\sigma^2 S_1^2 \frac{\partial^2 V}{\partial S_1^2} + \frac{1}{2}\sigma^2 S_2^2 \frac{\partial^2 V}{\partial S_2^2} - rV = 0$$

Because the payoff $V(T) = \max(S_1, S_2)$ scales linearly, it can be reduced into a 1D problem by looking at the ratio of the two stocks.

Let $z = \frac{S_2}{S_1}$ and define the price as $V(S_1, S_2, t) = S_1 f(z, t)$.

Calculate the partial derivatives of $V$ in terms of $f$ and substitute them back into the 2D Black-Scholes PDE:
- $V_{S_1}:$ with the product rule:
    $$\frac{\partial V}{\partial S_1} = f(z, t) + [S_1 \cdot \frac{\partial f}{\partial z} \cdot \frac{\partial z}{\partial S_1}]$$
    Since $\frac{\partial z}{\partial S_1} = -\frac{S_2}{S_1^2}$:
    $$V_{S_1} = f + S_1 \cdot f_z \cdot \left( -\frac{S_2}{S_1^2} \right) = f - \frac{S_2}{S_1} f_z = f - z f_z$$
- $V_{S_2}$:
$$\frac{\partial V}{\partial S_2} = S_1 \cdot \frac{\partial f}{\partial z} \cdot \frac{\partial z}{\partial S_2}$$ because $\frac{\partial z}{\partial S_2} = \frac{1}{S_1}$:
$$V_{S_2} = S_1 \cdot f_z \cdot \frac{1}{S_1} = f_z$$

- $V_{S_1 S_1}:$ differentiate $V_{S_1}$:
    $$V_{S_1 S_1} = \frac{\partial f}{\partial S_1} - \frac{\partial (z f_z)}{\partial S_1}$$
    Using the chain rule results:
    $$V_{S_1 S_1} = \left( f_z \cdot \frac{-S_2}{S_1^2} \right) - \left[ \left( \frac{-S_2}{S_1^2} \right) f_z + z \left( f_{zz} \cdot \frac{-S_2}{S_1^2} \right) \right]$$
    The first two terms cancel out:
    $$V_{S_1 S_1} = -z \cdot f_{zz} \cdot \left( -\frac{S_2}{S_1^2} \right) = \frac{z S_2}{S_1^2} f_{zz} = \frac{z (z S_1)}{S_1^2} f_{zz} = \frac{z^2}{S_1} f_{zz}$$

- $V_{S_2 S_2}:$
$$V_{S_2 S_2} = \frac{\partial}{\partial S_2} (f_z) = \frac{\partial f_z}{\partial z} \cdot \frac{\partial z}{\partial S_2} = f_{zz} \cdot \frac{1}{S_1} = \frac{f_{zz}}{S_1}$$


Put these into the 2D PDE:
$$\frac{\partial V}{\partial t} + r S_1 V_{S_1} + r S_2 V_{S_2} - r V + (\text{volatility terms}) = 0$$

Group all terms containing r:
$$r S_1 (f - z f_z) + r S_2 (f_z) - r (S_1 f)$$
$$r S_1 f - r S_1 z f_z + r S_2 f_z - r S_1 f$$
$r S_1 f$ and $-r S_1 f$ cancel out and $-r S_2 f_z + r S_2 f_z = 0$. The r cancels out from the equation.

The two volatility parts:
$$\frac{1}{2} \sigma^2 S_1^2 V_{S_1 S_1} + \frac{1}{2} \sigma^2 S_2^2 V_{S_2 S_2}$$
$$\frac{1}{2} \sigma^2 S_1^2 \left( \frac{z^2}{S_1} f_{zz} \right) + \frac{1}{2} \sigma^2 S_2^2 \left( \frac{f_{zz}}{S_1} \right)$$
$$\frac{1}{2} \sigma^2 S_1 z^2 f_{zz} + \frac{1}{2} \sigma^2 \frac{S_2^2}{S_1} f_{zz}$$

Since $S_2^2 = (z S_1)^2 = z^2 S_1^2$, the second term becomes $\frac{1}{2} \sigma^2 z^2 S_1 f_{zz}$.

Adding them together gives:
$$\sigma^2 S_1 z^2 f_{zz}$$

Plugging $V_t = S_1 f_t$, the zeroed $r$ terms, and the combined volatility terms back into the main equation:
$$S_1 f_t + 0 + \sigma^2 S_1 z^2 f_{zz} = 0$$

Divide the whole equation by $S_1$:
$$\frac{\partial f}{\partial t} + \sigma^2 z^2 \frac{\partial^2 f}{\partial z^2} = 0$$

To solve this using standard Finite Difference Methods, the coefficients should be constants (like the Heat Equation). For this, perform two substitutions: convert $z$ to log-moneyness and reverse the time:
- Log-Moneyness: $x = \ln(z)$. If $z = e^x$, then using the chain rule, the derivatives become:
$$\frac{\partial f}{\partial z} = \frac{1}{z} \frac{\partial g}{\partial x}$$
$$\frac{\partial^2 f}{\partial z^2} = \frac{1}{z^2} \left( \frac{\partial^2 g}{\partial x^2} - \frac{\partial g}{\partial x} \right)$$
- Time-to-maturity: $\tau = T - t$.  Standard Black-Scholes moves from maturity to today. By letting $\tau = T - t$, $\tau = 0$ is "today", and it moves toward the maturity date. This changes $\frac{\partial f}{\partial t}$ into $-\frac{\partial g}{\partial \tau}$

Leting $g(x, \tau) = f(e^x, T-\tau)$ and applying the chain rule, the final PDE becomes:$$\frac{\partial g}{\partial \tau} = \sigma^2 \left( \frac{\partial^2 g}{\partial x^2} - \frac{\partial g}{\partial x} \right)$$

Conditions for FDM:
* Initial condition at maturity ($\tau=0$): $g(x, 0) = \max(1, e^x)$
* Boundary as $x \to -\infty$ (Stock 1 dominates): $g \to 1$
* Boundary as $x \to \infty$ (Stock 2 dominates): $g \to e^x$

## Part c.) Solving with the Finite-Difference Method

In [9]:
# Analytical Solution

d1 = sigma * np.sqrt(T / 2.0)
analytical_price = 2.0 * S0 * norm.cdf(d1)
print("Analytical Price:", analytical_price)

Analytical Price: 111.24629160182849


In [10]:
# Explicit Finite Difference Method

# Grid setup
nx = 201 # Number of spatial steps
nt = 4000 # Number of time steps
x_min, x_max = -5.0, 5.0

dx = (x_max - x_min) / (nx - 1)
dt = T / (nt - 1)
x_grid = np.linspace(x_min, x_max, nx)

# Initial Condition at tau = 0 (maturity)
g = np.maximum(1.0, np.exp(x_grid))

for n in range(nt):
    g_new = np.zeros(nx)

    # Boundary Conditions
    g_new[0] = 1.0
    g_new[-1] = np.exp(x_max)

    # Interior points with central difference
    g_xx = (g[2:] - 2 * g[1:-1] + g[:-2]) / (dx**2)
    g_x = (g[2:] - g[:-2]) / (2 * dx)

    # Update the PDE
    g_new[1:-1] = g[1:-1] + dt * (sigma**2) * (g_xx - g_x)

    # Move to the next time step
    g = g_new

# Extract price at x=0
idx_0 = np.argmin(np.abs(x_grid))

fdm_price = S0 * g[idx_0]

print("FDM Price:", fdm_price)

FDM Price: 111.20222946393885


## d.) Monte Carlo Method

$N$ independent paths under the risk-neutral measure:
$$S_i(T) = S_0 \exp  (( r - \frac{\sigma^2}{2}) T + \sigma \sqrt{T} , Z_i ),
\quad Z_1, Z_2 \overset{\text{i.i.d.}} {\sim} N(0,1),$$

and estimate the price as
$$\hat{V}(0) = e^{-r T} \frac{1}{N} \sum_{k=1}^N \max \{S_1^{k} (T), S_2^{k} (T)\}.$$

In [11]:
Z1 = np.random.randn(N_sims)
Z2 = np.random.randn(N_sims)

# Simulate stock prices using the Geometric Brownian Motion formula
S1_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z1)
S2_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z2)

# Payoff is the maximum of the two stocks
payoffs = np.maximum(S1_T, S2_T)

# Discount the expected payoff back to t=0
mc_price = np.exp(-r * T) * np.mean(payoffs)
print("Monte Carlo Price:", mc_price)

Monte Carlo Price: 111.21998559541242


---

## AI Usage

- Gemini 3.1 Pro: simplifying the PDE, fixing numpy array indexing